In [ ]:
!pip install ragas==0.3.0

In [ ]:
from dotenv import load_dotenv

load_dotenv("/home/ubuntu/work/edu-src-all/.env")

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from ragas.metrics import (
    faithfulness,
    answer_relevancy,   # 생성된 답변이 원래 질문에 얼마나 적절한지 측정. 질문에 직접적으로 응답하는 답변은 높은 점수
    context_precision,  # 생성된 답변에 사용된 컨텍스트 정보가 얼마나 관련성이 있는지 측정. 불필요한 정보 없이 질문과 관련된 컨텍스트만 포함되어 있으면 높은 점수
    context_recall,     # 답변 생성에 필요한 모든 정보가 검색된 컨텍스트에 포함되어 있는지 측정. 질문에 답하는 데 필요한 정보가 모두 검색되면 높은 점수
    answer_correctness  # 생성된 답변이 참조 답변과 비교하여 얼마나 정확한지 평가, 참조 답변이 필요
)
from ragas.evaluation import evaluate
from datasets import Dataset
import pandas as pd

In [ ]:
def load_documents(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    return documents

In [ ]:
def split_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    splits = text_splitter.split_documents(documents)
    return splits

In [ ]:
def create_vectorstore(splits):
    embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(splits, embeddings)
    return vectorstore

In [ ]:
def create_rag_pipeline(vectorstore):
    llm = ChatOpenAI(model_name=os.getenv("OPENAI_DEFAULT_MODEL"), temperature=0)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True
    )
    return qa_chain

In [ ]:
def generate_test_questions(pdf_path, num_questions=5):
    documents = load_documents(pdf_path)
    llm = ChatOpenAI(model_name=os.getenv("OPENAI_DEFAULT_MODEL"), temperature=0.9)
    questions = []
    
    for i in range(num_questions):
        prompt = f"""
        다음 문서를 바탕으로 질문을 1개 생성해주세요.
        문서 내용: {documents[i % len(documents)].page_content}
        """
        question = llm.invoke(prompt).content
        questions.append(question)
    
    return questions

In [ ]:
pdf_path = "data/소나기.pdf"

documents = load_documents(pdf_path)
splits = split_documents(documents)

In [ ]:
vectorstore = create_vectorstore(splits)

In [ ]:
qa_chain = create_rag_pipeline(vectorstore)

In [ ]:
questions = generate_test_questions(pdf_path, 20)
questions

### Custom Metric

In [ ]:
from dataclasses import dataclass, field
from ragas.metrics.base import MetricWithLLM, SingleTurnMetric, MetricType
from ragas.callbacks import Callbacks
from ragas.dataset_schema import SingleTurnSample
from langchain.messages import HumanMessage, SystemMessage

import openai
import typing as t

In [ ]:
import os
from ragas.llms import LangchainLLMWrapper

llm = ChatOpenAI(temperature=0, model_name=os.getenv("OPENAI_DEFAULT_MODEL"))
evaluator_llm = LangchainLLMWrapper(llm)

In [ ]:

@dataclass
class HallucinationSeverityMetric(MetricWithLLM, SingleTurnMetric):
    name: str = "hallucination_severity"
    _required_columns: t.Dict[MetricType, t.Set[str]] = field(
        default_factory=lambda: {
            MetricType.SINGLE_TURN: {"user_input", "response", "retrieved_contexts"}
        }
    )

    async def _single_turn_ascore(
        self, sample: SingleTurnSample, callbacks: Callbacks
    ) -> float:
        prompt = f"""
        Context: {sample.retrieved_contexts}
        Answer: {sample.response}
        How severe is the hallucination? Compare context and answer. Score from 0 (no hallucination) to 1 (completely false).
        Only answer value.
        """

        messages = [
            SystemMessage(content="Evaluate hallucination severity."),
            HumanMessage(content=prompt)
        ]

        response = self.llm.langchain_llm.invoke(messages)
        try:
            hallucination_score = float(response.content.strip())
            if 0 <= hallucination_score <= 1:
                return hallucination_score
            else:
                return 0.0 
        except ValueError:
            return 0.0                       
        return float(response.content)

In [ ]:
hallucinationSeverityMetric = HallucinationSeverityMetric(llm=evaluator_llm)
sample = SingleTurnSample(
    user_input="파리는 어느 나라의 수도인가요?",
    response="파리는 프랑스의 수도입니다.",
    retrieved_contexts=["파리는 독일의 수도입니다. 독일은 유럽에 위치한 국가로, 파리는 세계적인 문화, 예술, 패션의 중심지로 알려져 있습니다."]
)
await hallucinationSeverityMetric.single_turn_ascore(sample)

In [ ]:
@dataclass
class ConcisenessMetric(MetricWithLLM, SingleTurnMetric):
    name: str = "conciseness"
    _required_columns: t.Dict[MetricType, t.Set[str]] = field(
        default_factory=lambda: {
            MetricType.SINGLE_TURN: {"user_input", "response", "retrieved_contexts"}
        }
    )

    async def _single_turn_ascore(
        self, sample: SingleTurnSample, callbacks: Callbacks
    ) -> float:
        context_length = len(" ".join(sample.retrieved_contexts).split())
        answer_length = len(sample.response.split())
        conciseness_score = max(0, min(1, 1 - (answer_length / (context_length + 1))))
        return conciseness_score

In [ ]:
concisenessMetric = ConcisenessMetric(llm=evaluator_llm)
sample = SingleTurnSample(
    user_input="파리는 어느 나라의 수도인가요?",
    response="파리는 프랑스의 수도입니다.",
    retrieved_contexts=["파리는 프랑스의 수도입니다. 프랑스는 유럽에 위치한 국가로, 파리는 세계적인 문화, 예술, 패션의 중심지로 알려져 있습니다."]
)
await concisenessMetric.single_turn_ascore(sample)

In [ ]:
@dataclass
class ContradictionDetectionMetric(MetricWithLLM, SingleTurnMetric):
    name: str = "contradiction_detection"
    _required_columns: t.Dict[MetricType, t.Set[str]] = field(
        default_factory=lambda: {
            MetricType.SINGLE_TURN: {"user_input", "response", "retrieved_contexts"}
        }
    )

    async def _single_turn_ascore(
        self, sample: SingleTurnSample, callbacks: Callbacks
    ) -> float:
        prompt = f"""
        Context: {sample.retrieved_contexts}
        Response: {sample.response}
        Does the response contradict the context? Answer with a score between 0 (no contradiction) and 1 (complete contradiction).
        Only answer value.
        """

        messages = [
            SystemMessage(content="Evaluate contradiction between context and response."),
            HumanMessage(content=prompt)
        ]

        response = self.llm.langchain_llm.invoke(messages)

        try:
            contradiction_score = float(response.content.strip())
            if 0 <= contradiction_score <= 1:
                return contradiction_score
            else:
                return 0.0 
        except ValueError:
            return 0.0

In [ ]:
contradictionDetectionMetric = ContradictionDetectionMetric(llm=evaluator_llm)
sample = SingleTurnSample(
    user_input="파리는 어느 나라의 수도인가요?",
    response="파리는 프랑스의 수도입니다.",
    retrieved_contexts=["파리는 독일의 수도입니다. 독일은 유럽에 위치한 국가로, 파리는 세계적인 문화, 예술, 패션의 중심지로 알려져 있습니다."]
)
await contradictionDetectionMetric.single_turn_ascore(sample)

In [ ]:
def evaluate_with_ragas_with_new_metrics(qa_chain, questions):
    evaluation_data = {
        "question": [],
        "answer": [],
        "contexts": []
    }
    
    evaluation_data["reference"] = []
    for question in questions:
        result = qa_chain(question)
        answer = result["result"]
        contexts = [doc.page_content for doc in result["source_documents"]]
        
        evaluation_data["question"].append(question)
        evaluation_data["answer"].append(answer)
        evaluation_data["contexts"].append(contexts)
        
        evaluation_data["reference"].append(contexts[0] if contexts else "")
    
    eval_dataset = Dataset.from_dict(evaluation_data)
    
    metrics = []

    metrics.append(faithfulness)
    metrics.append(answer_relevancy)
    metrics.append(context_precision)
    metrics.append(context_recall)

    metrics.append(HallucinationSeverityMetric(llm=evaluator_llm))
    metrics.append(ConcisenessMetric(llm=evaluator_llm))
    metrics.append(ContradictionDetectionMetric(llm=evaluator_llm))
    
    results = evaluate(
        eval_dataset,
        metrics=metrics
    )
    
    results_df = pd.DataFrame({
        "question": evaluation_data["question"],
        "answer": evaluation_data["answer"]
    })
    
    for metric in metrics:
        results_df[metric.name] = results[metric.name]
    
    return results_df


In [ ]:
results = evaluate_with_ragas_with_new_metrics(qa_chain, questions)

In [ ]:
print("평가 결과:")
print(results)

metric_columns = [col for col in results.columns if col not in ["question", "answer"]]
if metric_columns:
    avg_scores = results[metric_columns].mean()
    print("\n평균 점수:")
    print(avg_scores)
